# Slumber

> Sleepy datasets and helpers

In [ ]:
#| default_exp slumber

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import pyedflib, edfio, torch, zarr, warnings, datetime as dt, torch.nn.functional as F, dask.array as da, numpy as np, pandas as pd
from torch.utils.data import Dataset
from scipy.signal import resample
import torchaudio.functional as F_audio
from typing import Union
from pathlib import Path
from mne.io import read_raw_edf

from sleepjepa.data_preprocessing import calculate_samples_mp, interpolate_nan_clip, check_hypnograms_mp
from sleepjepa.signal import iir_filter, iqr_normalization, resample_waveform
from scipy.ndimage import median_filter


## Constants

In [ ]:
#| export
# wsc channels for training, note it does not have C4_M1
WSC_CHANNELS = [['ECG'], ['E1'], ['cchin_l', 'chin'], ['C3_M2'], ['spo2'], ['thorax'], ['abdomen']]

# apples channels for training, double checked and in the APPLES paper, LOC is referenced with M2
## see: https://academic.oup.com/sleep/article/34/3/303/2433804 and https://gitlab-scm.partners.org/zzz-public/nsrr/-/tree/master/studies/apples
APPLES_CHANNELS = [["ECG"], ["LOC"], ["EMG"], ["C4_M1", "C3_M2"], ["SpO2"], ["thorax"], ["abdomen"]]

# mesa channels for training, note that it does not have C3_M2, EEG3 == C4_M1
## note also that mesa refereences E1-fpz! not E!-M@
MESA_CHANNELS = [["EKG"], ["EOG-L"], ["EMG"], ["EEG3"], ["SpO2"], ["Thor"], ["Abdo"]]

# shhs channels for training 
## note that EOG(L) is referenced with PG1 (ground)
## EEG is C4_M1 and EEG(sec) is C3_M2
SHHS_CHANNELS = [["ECG"], ["EOG(L)"], ["EMG"], ["EEG", "EEG(sec)"], ["SaO2"], ["THOR RES"], ["ABDO RES"]]

# HSP
## EOG(L) was mapped from E1-M2, check the build_hsp_zarrs.py job
HSP_CHANNELS = [['ECG', 'ECG (LL-RA)'], ['EOG(L)', 'E1-AVG'], ['EMG', 'EMG (1-2)', 'EMG (1-3)', 'Chin3'], ['C4-M1', 'C3-M2', 'C4-AVG', 'C3-AVG'], ['SaO2', 'SPO2'], ['THOR RES', 'Chest'], ['ABDO RES', 'Abd']]

# most of these weere derived in the zarr job file
MROS_CHANNELS = [['ECG (L-R)'], ['E1-M2'], ['EMG (L-R)'], ["C4-M1", "C3-M2"], ['SaO2', 'SpO2'], ['Thoracic', 'Chest'], ['Abdominal', 'ABD']]

MNC_CHANNELS = [['ECG'], ['LOC'], ['EMG', 'cchin', 'chin', 'cchin_l'], ['C4-M1'], ['spo2'], ['thorax'], ['abdomen']]

ALL_CHANNELS = [['ECG', 'ECG (LL-RA)', 'EKG', 'ECG (L-R)', 'ECG2'],
 ['EOG(L)', 'EOG-L', 'E1', 'LOC', 'E1-M2', 'E1-AVG'],
 ['EMG', 'cchin_l', 'chin', 'EMG (L-R)', 'Chin 1-Chin 2', 'EMG (1-2)', 'EMG (1-3)', 'Chin3', 'cchin'],
 ['C4-M1', 'C4_M1', 'EEG', 'EEG3', 'C3-M2', 'C3_M2', 'EEG(sec)', 'C4-AVG', 'C3-AVG'],
 ['SaO2', 'SpO2', 'spo2', 'SPO2'],
 ['THOR RES', 'Thor', 'thorax', 'Thoracic', 'Chest'],
 ['ABDO RES', 'abdomen', 'Abdo', 'Abdominal', 'ABD', 'Abd']]

# derived from https://www.researchsquare.com/article/rs-6307069/v1
ALL_FREQUENCY_FILTERS = {'ECG':[None,0.3], # highpass 0.3 Hz
                         'ECG (LL-RA)':[None,0.3],
                         'EKG':[None,0.3],
                         'ECG (L-R)':[None,0.3],
                         'ECG2':[None,0.3],

                         'EOG(L)':[0.3,45],
                         'EOG-L':[0.3,45],
                         'E1':[0.3,45],
                         'LOC':[0.3,45],
                         'E1-M2':[0.3,45],
                         'E1-AVG':[0.3,45],

                         'EMG':[None,10],
                         'cchin_l':[None,10],
                         'chin':[None,10],
                         'EMG (L-R)':[None,10],
                         'Chin 1-Chin 2':[None,10],
                         'EMG (1-2)':[None,10],
                         'EMG (1-3)':[None,10],
                         'Chin3':[None,10],
                         'cchin':[None,10],

                         'C4-M1':[0.3,45],
                         'C4_M1':[0.3,45],
                         'EEG':[0.3,45],
                         'EEG3':[0.3,45],
                         'C3-M2':[0.3,45],
                         'C3_M2':[0.3,45],
                         'EEG(sec)':[0.3,45],
                         'C4-AVG':[0.3,45],
                         'C3-AVG':[0.3,45],

                        #  'SaO2':[0.05,5],
                        #  'SpO2':[0.05,5],
                        #  'spo2':[0.05,5],
                        #  'SPO2':[0.05,5],

                         'THOR RES':[0.1,15],
                         'Thor':[0.1,15],
                         'thorax':[0.1,15],
                         'Thoracic':[0.1,15],
                         'Chest':[0.1,15],

                         'ABDO RES':[0.1,15],
                         'abdomen':[0.1,15],
                         'Abdo':[0.1,15],
                         'Abdominal':[0.1,15],
                         'ABD':[0.1,15],
                         'Abd':[0.1,15]}

VOLTAGE_CHANNELS = ['ECG','ECG (LL-RA)','EKG','ECG (L-R)', 'ECG2','EOG(L)','EOG-L','E1','LOC','E1-M2','E1-AVG','EMG','cchin_l','chin','EMG (L-R)','Chin 1-Chin 2','EMG (1-2)','EMG (1-3)','Chin3','cchin','C4-M1','C4_M1','EEG','EEG3','C3-M2','C3_M2','EEG(sec)','C4-AVG','C3-AVG']
SPO2_CHANNELS = ['SaO2', 'SpO2', 'spo2', 'SPO2']
# all_channels = [[] for _ in range(7)]
# for channel_group in [SHHS_CHANNELS, MESA_CHANNELS, WSC_CHANNELS, APPLES_CHANNELS, HSP_CHANNELS, MROS_CHANNELS, AIM_AHI]:
#     for i, channel in enumerate(channel_group):
#         all_channels[i] = list(set(all_channels[i]) | set(channel))

# all_channels

## EDF Helpers

### Read EDFs

In [ ]:
#| export
def read_edf(file_path, # file path of edf
             channels=None, # channels in edf to read, will raise warning if channels do not exist
             frequency=None # frequency to resample all signals to
             )->Union[list, dict]: # tuple of signals and header dictionary
    """
    Function to read an edf file and return a list of signals and header with the option to resample to a passed frequency
    """
    pyedflib.close_file(0) # ensure files are closed 
    with pyedflib.EdfReader(file_path) as f:
        num_channels = f.signals_in_file
        if channels is not None:
            available_channels = [channel.upper() for channel in f.getSignalLabels()]
            channels = [channel.upper() for channel in channels]
            if len(set(channels) - set(available_channels)) != 0:
                warnings.warn(f'Missing channels {set(channels) - set(available_channels)}')
            channels_to_get = list(set(channels) & set(available_channels))
            channels_idxs = [available_channels.index(c) for c in channels_to_get]
        else:
            channels_idxs = range(num_channels)
        header = f.getHeader()
        header['Duration'] = int(f.getFileDuration())
        header['SignalHeaders'] = [f.getSignalHeaders()[c] for c in channels_idxs]
        signals = []
        if frequency is not None:
            required_length = header['Duration'] * frequency
        for c in channels_idxs:
            signal = f.readSignal(c, digital=False)
            if frequency is not None and len(signal) != required_length:
                signal = resample(signal, required_length)
                header['SignalHeaders'][c]['sample_rate'] = frequency
                header['SignalHeaders'][c]['sample_frequency'] = frequency
            signals.append(signal)
    return signals, header

In [ ]:
#| export
def read_edf_mne(file_path, # file path of edf
             channels=None, # channels in edf to read, will raise warning if channels do not exist
             frequency=None # frequency to resample all signals to
             )->Union[list, dict]: # tuple of signals and header dictionary
    """
    function to read edf with mne library
    i dont recommend using this. Use edfio instead.
    """
    raw_edf = read_raw_edf(file_path)
    signal_labels = raw_edf.ch_names
    num_channels = len(signal_labels)
    if channels is not None:
        available_channels = [channel.upper() for channel in signal_labels]
        channels = [channel.upper() for channel in channels]
        if len(set(channels) - set(available_channels)) != 0:
            warnings.warn(f'Missing channels {set(channels) - set(available_channels)}')
        channels_to_get = list(set(channels) & set(available_channels))
        channels_idxs = [available_channels.index(c) for c in channels_to_get]
    else:
        channels_idxs = range(num_channels)
    info_dict = dict(raw_edf.info)
    header = {'technician': info_dict.get('experimenter'),
        'recording_additional': '',
        'patientname': info_dict.get('subject_info').get('last_name'),
        'patient_additional': '',
        'patientcode': '',
        'equipment': '',
        'admincode': '',
        'sex': 'Female' if info_dict.get('subject_info').get('sex') == 0 else 'Male',
        'startdate': info_dict.get('meas_date'),
        'birthdate': info_dict.get('subject_info').get('birthday'),
        'gender': 'Female' if info_dict.get('subject_info').get('sex') == 0 else 'Male',
        'Duration':int(len(raw_edf) / info_dict.get('sfreq')),
        'SignalHeaders':[{'label':i['ch_name'], 'sample_frequency':info_dict.get('sfreq')} for i in info_dict.get('chs')]
    }
    signals = []
    if frequency is not None:
        required_length = header['Duration'] * frequency
    for c in channels_idxs:
        signal = raw_edf[c][0][0]
        if frequency is not None and len(signal) != required_length:
            signal = resample(signal, required_length)
        signals.append(signal)
    return signals, header

In [ ]:
#| export
def read_edf_edfio(file_path, # file path of edf
             channels=None, # channels in edf to read, will raise warning if channels do not exist
             frequency=None # frequency to resample all signals to
             )->Union[list, dict]: # tuple of signals and header dictionary
    """
    function to read edfs with edfio
    """
    f = edfio.read_edf(file_path, lazy_load_data=True)
    num_channels = f.num_signals
    if channels is not None:
        available_channels = [channel.upper() for channel in f.labels]
        channels = [channel.upper() for channel in channels]
        if len(set(channels) - set(available_channels)) != 0:
            warnings.warn(f'Missing channels {set(channels) - set(available_channels)}')
        channels_to_get = list(set(channels) & set(available_channels))
        channels_idxs = [available_channels.index(c) for c in channels_to_get]
    else:
        channels_idxs = range(num_channels)
    try:
      start_datetime = dt.datetime.combine(f.startdate, f.starttime)
    except:
      start_datetime = None
    header = {'technician': '' if f.recording.get_subfield(2) == 'X' else f.recording.get_subfield(2),
            'recording_additional': '',
            'patientname': '' if f.patient.get_subfield(3) == 'X' else f.patient.get_subfield(3),
            'patient_additional': '',
            'patientcode': '',
            'equipment': '',
            'admincode': '',
            'sex': '' if f.patient.get_subfield(1) == 'X' else f.patient.get_subfield(1).replace('M', 'Male').replace('F', 'Female'),
            'startdate': start_datetime,
            'birthdate': f.patient.get_subfield(2).lower().replace('-',' ') if f.patient.get_subfield(2) != 'X' else '',
            'gender': '' if f.patient.get_subfield(1) == 'X' else f.patient.get_subfield(1).replace('M', 'Male').replace('F', 'Female'),
            'Duration': int(f.duration),
            'SignalHeaders':[{'label':f.signals[i].label, 
                            'dimension':f.signals[i].physical_dimension,
                            #'sample_rate':f.signals[i].sampling_frequency if frequency is None else frequency,
                            'sample_frequency':f.signals[i].sampling_frequency if frequency is None else frequency,
                            'physical_max': f.signals[i].physical_max,
                            'physical_min': f.signals[i].physical_min,
                            'digital_max': f.signals[i].digital_max,
                            'digital_min': f.signals[i].digital_min,
                            'prefilter': f.signals[i].prefiltering,
                            'transducer': f.signals[i].transducer_type
                            } for i in channels_idxs]
    }
    signals = []
    if frequency is not None:
        required_length = header['Duration'] * frequency
    for c in channels_idxs:
        signal = f.signals[c].data
        if frequency is not None and len(signal) != required_length:
            signal = resample(signal, required_length)
        signals.append(signal)
    return signals, header

### Read Hypnograms

In [ ]:
#| export
def read_hypnogram(file, # file path of the hypnogram csv
                   epoch_length = None # epoch length of the hypnogram measurements, if passed will repeat this many times at each element
                   )->np.array: # numpy array of hypnogram
     """
     Function that reads a hypnogram csv and returns a numpy array of the hypnogram with optional repeats
     """
     y = pd.read_csv(file, header=None, names=[1])[1].to_numpy()
     if epoch_length is not None:
         y = y.repeat(epoch_length)
     return y

### EDFs to Zarr

In [ ]:
#| export
def edf_signals_to_zarr(edf_file_path, write_data_dir, overwrite=False, channels=None, channel_name_map=None, frequency=None, hyp_epoch_length=30, hyp_data_dir=None):
    """
    Function that converts an edf to a zarr file
    """
    if channel_name_map is None:
        channel_name_map = {}
    edf_file_path = Path(edf_file_path)
    try:
        signals, header = read_edf(str(edf_file_path), channels=channels, frequency=frequency)
    except Exception as e:
        # try with edfio library, mne library is meh, because mne resamples data to highest frequency
        # then it adjusts units. edfio and pyedflib do not do this.
        signals, header = read_edf_edfio(str(edf_file_path), channels=channels, frequency=frequency)
    store = zarr.storage.LocalStore(str(Path(write_data_dir)/edf_file_path.stem) + '.zarr')
    if overwrite:
        root_grp = zarr.group(store, overwrite=True)
    else:
        # this should not replace data, so we can continue to append to this group.
        root_grp = zarr.group(store, overwrite=False)
    signal_headers = header.pop('SignalHeaders')
    root_grp.attrs['header'] = {k:str(v) for k,v in header.items()}
    # get y/sleep staging data
    try:
        # try to get hypnogram data if it exists
        if hyp_data_dir is not None:
            hyp_data_dir = Path(hyp_data_dir)
            hyp_file_path = hyp_data_dir/Path(edf_file_path.stem + "-hyp.csv")
        else:
            hyp_file_path = edf_file_path.parent/Path(edf_file_path.stem + "-hyp.csv") # [V] V is variable per file and does not necessarily == patch_num
        
        if frequency is not None:
            y = read_hypnogram(hyp_file_path, epoch_length=(hyp_epoch_length*frequency))
        else:
            y = read_hypnogram(hyp_file_path, epoch_length=(hyp_epoch_length))

        if frequency is not None and len(y) < len(signals[-1]):
            # hypnogram is shorter, trim signals
            signals = signals[:, :len(y)] # [n_vars x T]
        elif frequency is not None and len(y) > len(signals[-1]):
            # signals are longer, trim hypnogram
            y = y[:len(signals[-1])] # [T]
    except:
        y = None
        pass
    # write signals
    for i, h in zip(signals, signal_headers):
        a = da.from_array(np.array(i, dtype=np.float16), chunks='auto') # read using dask
        name = h['label'] if h['label'] not in channel_name_map else channel_name_map[h['label']]
        h['mapped_label'] = name
        a.to_zarr(url=store, component=name, compute=True) # convert to zarr format
        root_grp[name].attrs['signal_header'] = h # assign metadata

    # write hypnogram
    if y is not None:
        a = da.from_array(y, chunks='auto')
        a.to_zarr(url=store, component='hypnogram', compute=True)
    zarr.consolidate_metadata(store)
    return root_grp

## Datasets

### Self Supervised Dataset

In [ ]:
#| export
def trim_wake_epochs_from_hypnogram(hypnogram, padding_mask=-100):
    """
    Function to trim wake epochs (if wake is the largest class) from hypnograms
    This function trims the wake epochs from the beginning and/or end of the hypnogram

    Adapted from Phan et al L-SeqSleepNet
    """
    # Check if Wake (stage 0) is the largest class
    idx, counts = hypnogram.unique(return_counts=True)
    counts = counts.numpy()
    if 0 not in idx:
        return hypnogram
    wake_idx = torch.where(idx==0)[0][0].item()
    n_wakes = counts[wake_idx].item()
    hypnogram = hypnogram.numpy()
    if n_wakes > np.max(counts[wake_idx+1:]):
        second_largest = np.max(counts[wake_idx+1:])
        
        # Create boolean array for Wake indices (True where stages == 0)
        W_ind = (hypnogram == 0)
        
        # Find first transition from/to Wake
        transitions = np.diff(W_ind.astype(int))
        last_evening_W_index = np.where(transitions != 0)[0][0]
        
        # Calculate number of evening Wake epochs
        num_evening_W = last_evening_W_index + 1 if hypnogram[0] == 0 else 0
        
        # Find last transition from/to Wake
        first_morning_W_index = np.where(transitions != 0)[0][-1] + 1
        num_morning_W = len(hypnogram) - first_morning_W_index
        
        nb_pre_post_sleep_wake_eps = num_evening_W + num_morning_W
        
        if nb_pre_post_sleep_wake_eps > second_largest:
            total_W_to_remove = nb_pre_post_sleep_wake_eps - second_largest
            
            if num_evening_W >= total_W_to_remove:
                # Remove from beginning only
                hypnogram[:total_W_to_remove] = padding_mask
            else:
                # Remove from both ends
                evening_W_to_remove = num_evening_W
                morning_W_to_remove = total_W_to_remove - evening_W_to_remove
                hypnogram[:evening_W_to_remove] = padding_mask
                hypnogram[-morning_W_to_remove:] = padding_mask

    return torch.from_numpy(hypnogram)


def remove_wake_epochs_from_signals(X, hypnogram, resampled_hypnogram_length, padding_mask=-100):
    """
    Function to trim wake epochs (if wake is the largest class) from signals

    X: bs, channels, seq_len
    hypnogram: bs, seq_len / resampled_hypnogram_length
    sequence_padding_mask: bs, seq_len
    """
    # find invalid exact timepoints
    invalid_segments = hypnogram.repeat_interleave(resampled_hypnogram_length) == padding_mask
    X = X[:, invalid_segments]
    return X


### Sleep Stage Self Supervised Dataset

In [ ]:
#| export
class SelfSupervisedHypnogramTimeDataset(Dataset):
    def __init__(self, 
                 zarr_files, # zarr files that include samples
                 channels, # channels to use
                 max_seq_len_sec, # maximum sequence length (in seconds) to use (this is especially relevant when you are returning both stft and raw ts data to keep them in sync)
                 sample_seq_len_sec, # if no sample_df, generate sequences of this length in seconds as one sample
                 sample_stride_sec, #  if no sample_df, seconds of overlap for samples from the same array, if seq_len_seconds == overlap_seconds, there is no overlap
                 frequency, # frequency of underlying data
                 min_seq_len_sec=None, # minimum sequence length (in seconds) to use
                 start_offset_sec=0, # number of seconds to exclude from beginning of sleep studies
                 trim_wake_epochs=True, # indicator to trim wake epochs from hypnograms, if it is the largest class
                 include_partial_samples=True, # indicator to include data from partial samples when return_full_length is false
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 return_hypnogram_every_sec=30, # integer value indicating the step in indexing in seconds
                 hypnogram_padding_mask=-100, # padded value to add to target and indice to ignore when computing loss
                 hypnogram_frequency=1, # frequency of underlying y hypnogram data
                 butterworth_filters=ALL_FREQUENCY_FILTERS, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # if not none, will apply median filter with kernel size
                 voltage_channels=None, # if not None, these channels units will be looked at and changed to microvolts from mv uv etc.
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 return_hyponogram=True, # indicator to return the hypnogram, this could be used for a classification task, otherwise will return X,X
                 normalize_signals=True, # indicator to normalize signals
                 constant_nan_tolerance=1.0, # tolerance for nan values in signals
                 constant_channels=SPO2_CHANNELS, # channels to check for constant values
                 hypnogram_required_stages=[0,1,2,3,4], # hypnogram stages that must be present in a sample
                 hypnogram_constant_tolerance=1.0 # tolerance for constant values in hypnogram
                 ):
        self.max_seq_len = max_seq_len_sec*frequency
        self.max_seq_len_sec = max_seq_len_sec
        self.min_seq_len_sec = min_seq_len_sec
        self.include_partial_samples = include_partial_samples
        self.zarr_files = zarr_files
        self.channels = channels
        self.channels_has_dim = any(isinstance(i, list) for i in self.channels)
        self.sample_seq_len_sec = sample_seq_len_sec
        self.sample_seq_len = sample_seq_len_sec * frequency
        self.frequency = frequency
        self.sample_stride_sec = sample_stride_sec
        self.clip_interpolations = clip_interpolations
        self.return_hypnogram_every_sec = return_hypnogram_every_sec
        self.hypnogram_padding_mask = hypnogram_padding_mask
        self.hypnogram_frequency = hypnogram_frequency
        self.start_offset_sec = start_offset_sec
        self.trim_wake_epochs = trim_wake_epochs
        self.return_hyponogram = return_hyponogram
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.voltage_channels = voltage_channels if voltage_channels is not None else []
        self.normalize_signals = normalize_signals
        self.constant_nan_tolerance = constant_nan_tolerance
        self.constant_channels = constant_channels
        self.hypnogram_required_stages = hypnogram_required_stages
        self.hypnogram_constant_tolerance = hypnogram_constant_tolerance
        if sample_df is None:
            print(f"Calculating samples with {sample_seq_len_sec} sec length and {sample_stride_sec} sec stride with {max_seq_len_sec} sec max")
            self.sample_df, self.total_samples = calculate_samples_mp(zarr_files, channels=channels, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, frequency=frequency, start_offset_sec=start_offset_sec, stride_sec=sample_stride_sec, include_partial_samples=include_partial_samples, min_seq_len_sec=min_seq_len_sec, constant_nan_tolerance=constant_nan_tolerance, constant_channels=constant_channels)
            if self.return_hyponogram and (self.hypnogram_required_stages is not None and len(self.hypnogram_required_stages) > 0) or (self.hypnogram_constant_tolerance < 1.0):
                valid_files = check_hypnograms_mp(self.sample_df['file'].unique(), required_stages=self.hypnogram_required_stages, constant_tolerance=self.hypnogram_constant_tolerance)
                self.sample_df = self.sample_df[self.sample_df['file'].isin(valid_files)].copy().reset_index(drop=True)
                self.total_samples = len(self.sample_df)
        else:
            assert 'file' in sample_df, "The `sample_df` must have a column `file` with the zarr file path."
            missing_in_zarrs = len(set(sample_df['file']) - set(zarr_files))
            missing_in_df = len(set(zarr_files) - set(sample_df['file']))
            if missing_in_zarrs > 0 or missing_in_df > 0:
                warnings.warn(f"There are {missing_in_zarrs} `zarr_files` not in the `sample_df` and {missing_in_df} files in the `sample_df` that arent in the `zarr_files`, they will be ignored.")
            sample_df = sample_df.loc[sample_df['file'].isin(zarr_files)].copy().reset_index(drop=True)
            self.sample_df = sample_df
            self.total_samples = len(sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample['file'], mode='r')

        signals = []
        channels = []
        if self.channels_has_dim:
            avail_channels = list(root_grp.array_keys())
            for p in self.channels:
                channels.append(next(x for x in p if x in avail_channels))
        else:
            channels = self.channels
        for channel in channels:
            channel_frequency = int(root_grp[channel].attrs.asdict()['signal_header']['sample_frequency'])
            channel_dimension = root_grp[channel].attrs.asdict()['signal_header']['dimension']
            if channel_frequency != self.frequency:
                start_idx, end_idx = int(sample['start_idx'] / self.frequency * channel_frequency), int(sample['end_idx'] / self.frequency * channel_frequency)
            else:
                start_idx, end_idx = sample['start_idx'], sample['end_idx']
            temp = np.array(root_grp[channel][start_idx:end_idx], dtype=np.float32)
            if channel_frequency != self.frequency:
                temp = resample_waveform(temp, channel_frequency, self.frequency, is_spo2=channel in SPO2_CHANNELS)
            if self.median_filter_kernel_size is not None:
                temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
            if self.butterworth_filters is not None and channel in self.butterworth_filters:
                freq_range = self.butterworth_filters[channel]
                btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                temp = iir_filter(temp, freq_range=freq_range, btype=btype, fs=self.frequency)
            if channel in self.voltage_channels:
                if channel_dimension.lower() == 'mv':
                    temp = temp * 1e3
            if self.clip_interpolations is not None and channel in self.clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
            else:
                temp = interpolate_nan_clip(temp)
            if self.normalize_signals:
                temp = iqr_normalization(temp, is_spo2=channel in SPO2_CHANNELS)
            signals.append(temp)
        X = torch.from_numpy(np.stack(signals, dtype=np.float32))
        
        if self.return_hyponogram:
            if self.hypnogram_frequency == self.frequency:
                hypnogram = torch.from_numpy(np.array(root_grp['hypnogram'][sample['start_idx']:sample['end_idx']], dtype=np.int64))
            else:
                hyp_start_idx = int((sample['start_idx'] / self.frequency) * self.hypnogram_frequency)
                hyp_end_idx = int((sample['end_idx'] / self.frequency) * self.hypnogram_frequency)
                hypnogram = np.array(root_grp['hypnogram'][hyp_start_idx:hyp_end_idx], dtype=np.int64)
                repeat_factor = self.frequency // self.hypnogram_frequency
                hypnogram = torch.from_numpy(np.repeat(hypnogram, repeat_factor))

                signal_length = X.shape[-1]
            
            if hypnogram.shape[-1] < signal_length:
                hypnogram = F.pad(hypnogram, (0, signal_length - len(hypnogram)), 'constant', value=self.hypnogram_padding_mask)
            elif hypnogram.shape[-1] > signal_length:
                hypnogram = hypnogram[:signal_length]
            
            samples_per_epoch = int(self.return_hypnogram_every_sec * self.frequency)
            hypnogram = hypnogram[::samples_per_epoch]

            assert hypnogram.max() <= 4 and (hypnogram.min() == self.hypnogram_padding_mask or hypnogram.min() >= 0), "There are values greater than 4 in the hypnogram or less than 0 that are not the padding mask. This is unexpected. 0-4 are the only valid values indicating wake, n1, n2, n3, rem, and a padding mask."
        if self.trim_wake_epochs:
            resampled_hypnogram_length = int(self.return_hypnogram_every_sec * self.frequency)
            hypnogram = trim_wake_epochs_from_hypnogram(hypnogram, padding_mask=self.hypnogram_padding_mask)
            X = remove_wake_epochs_from_signals(X, hypnogram=hypnogram, resampled_hypnogram_length=resampled_hypnogram_length, padding_mask=self.hypnogram_padding_mask)
        if self.return_hyponogram:
            Y = hypnogram
        else:
            Y = X

        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        if torch.isnan(Y).any():
            warnings.warn(f"Y has nan values, sample_idx: {idx}")
        return X, Y, idx

In [ ]:
#| export
def padded_tensor_sequence_collate(batch, max_seq_len, frequency, return_hypnogram=False, hypnogram_frequency=1, X_pad_value = 0., hypnogram_padding_mask = -100):
    """
    Collate function for variable length sequences with padding.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (channels, seq_len)
            Y: Target tensor (channels, seq_len) or hypnogram
            
    Returns:
        X: Tensor containing the batch of sequences
        Y: Tensor containing the batch of targets
    """
    # Separate X and Y from batch
    X, Y, idx = zip(*batch)
    
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension
    
    idx = torch.tensor(idx, dtype=torch.int64)
    X = [F.pad(i, (0, max_seq_len - i.shape[-1]), 'constant', value=X_pad_value) for i in X]
    X = torch.stack(X, dim=0).to(torch.float32)
    if return_hypnogram:
        expected_hyp_len = int((X.shape[-1] / frequency) * hypnogram_frequency)
        Y = [F.pad(i, (expected_hyp_len - i.shape[-1],), 'constant', value=hypnogram_padding_mask) for i in Y]
        Y = torch.stack(Y).to(torch.int64)
    else:
        # SS X == Y
        Y = [F.pad(i, (0, max_seq_len - i.shape[-1]), 'constant', value=X_pad_value) for i in Y]
        Y = torch.stack(Y).to(torch.float32)
    return X, Y, idx

In [ ]:
#| export
def nested_tensor_collate(batch):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (channels, seq_len)
            Y: single outcome (bs)
            idx: optional index of the sample in the dataset
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
        Y: Tensor containing the batch of targets
        idx: optional tensor of indices
    """
    # Separate X and Y from batch
    X, Y, *idx = zip(*batch)
    
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension
    X = [i.transpose(0,1) for i in X]

    X_nested = torch.nested.as_nested_tensor(X, layout=torch.jagged)
    Y = torch.stack(Y, dim=0).squeeze(-1)
    X_nested = X_nested.transpose(1,2) # transpose back to the original shape
    if len(idx) > 0:
        idx = idx[0]
        idx = torch.tensor(idx, dtype=torch.int64).squeeze(-1)
        return X_nested, Y, idx
    else:
        return X_nested, Y

In [ ]:
#| export
def nested_tensor_sequence_collate(batch):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (channels, seq_len)
            Y: Target tensor (channels, seq_len) or hypnogram
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
        Y: Tensor containing the batch of targets
    """
    # Separate X and Y from batch
    X, Y, idx = zip(*batch)
    
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension
    X = [i.transpose(0,1) for i in X]
    Y = [i.transpose(0,1) if i.dim() == 2 else i for i in Y]

    X_nested = torch.nested.as_nested_tensor(X, layout=torch.jagged)
    Y_nested = torch.nested.as_nested_tensor(Y, layout=torch.jagged)
    X_nested = X_nested.transpose(1,2) # transpose back to the original shape
    idx = torch.tensor(idx, dtype=torch.int64)
    if Y_nested.dim() == 3:
        Y_nested = Y_nested.transpose(1,2)
    return X_nested, Y_nested, idx

In [ ]:
#| export
def nested_tensor_sequence_multi_label_collate(batch):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (channels, seq_len) (nested tensor)
            Y: Tensor of shape (n_events)
            time: Tensor of shape (n_events)
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
        Y: Tensor containing the batch of targets
        time: Tensor containing the batch of times
    """
    # Separate X and Y from batch
    cols = list(zip(*batch))
    
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension
    X = [i.transpose(0,1) for i in cols[0]]

    X_nested = torch.nested.as_nested_tensor(X, layout=torch.jagged)
    X_nested = X_nested.transpose(1,2) # transpose back to the original shape
    out = [X_nested]
    for field in cols[1:]:
        out.append(torch.stack(field, dim=0).squeeze(-1))
    return tuple(out)

In [ ]:
#| export
def padded_tensor_sequence_multi_label_collate(batch, max_seq_len, X_pad_value = 0.):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tuples (X, Y) where:
            X: Tensor of shape (channels, seq_len) (nested tensor)
            Y: Tensor of shape (n_events)
            time: Tensor of shape (n_events)
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
        Y: Tensor containing the batch of targets
        time: Tensor containing the batch of times
    """
    # Separate X and Y from batch
    X, Y, time = zip(*batch)
    X = [F.pad(i, (0, max_seq_len - i.shape[-1]), 'constant', value=X_pad_value) for i in X]
    X = torch.stack(X, dim=0).to(torch.float32)
    
    Y = torch.stack(Y, dim=0).squeeze(-1)
    time = torch.stack(time, dim=0).squeeze(-1)
    return X, Y, time

### Single Outcome Dataset

In [ ]:
#| export
class SingleOutcomeDataset(Dataset):
    def __init__(self, 
                 zarr_files, # zarr files that include samples
                 channels, # channels to use
                 max_seq_len_sec, # maximum sequence length (in seconds) to use (this is especially relevant when you are returning both stft and raw ts data to keep them in sync)
                 sample_seq_len_sec, # if no sample_df, generate sequences of this length in seconds as one sample
                 sample_stride_sec, #  if no sample_df, seconds of overlap for samples from the same array, if seq_len_seconds == overlap_seconds, there is no overlap
                 y_outcome_df, # file path containing values for outcome of interest
                 min_seq_len_sec=None, # minimum sequence length (in seconds) to use
                 trim_wake_epochs=True, # indicator to trim wake epochs from hypnograms, if it is the largest class
                 return_hypnogram_every_sec=30, # integer value indicating the step in indexing in seconds
                 hypnogram_padding_mask=-100, # padded value to add to target and indice to ignore when computing loss
                 hypnogram_frequency=125, # frequency of underlying y hypnogram data
                 y_mapping_column='filepath', # column mapping corresponding outcome value to zarr file
                 y_outcome='', # outcome column in the y file path
                 y_time_column=None, # column in the y file path that contains the time of the event or censored
                 y_demographic_columns=None, # list of demographic columns to return as part of the outcome
                 y_demographic_norm_stats={}, # dictionary of demographic columns to normalize with mean and std
                 include_partial_samples=True, # indicator to include data from partial samples when return_full_length is false
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 start_offset_sec=0, # number of seconds to exclude from beginning of sleep studies
                 frequency=128, # frequency of underlying data
                 butterworth_filters=ALL_FREQUENCY_FILTERS, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # if not none, will apply median filter with kernel size
                 voltage_channels=None, # if not None, these channels units will be looked at and changed to microvolts from mv uv etc.
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 normalize_signals=True, # indicator to normalize signals
                 constant_nan_tolerance=1.0, # tolerance for nan values in signals
                 constant_channels=SPO2_CHANNELS, # channels to check for constant values
                 ):
        self.max_seq_len = max_seq_len_sec*frequency
        self.max_seq_len_sec = max_seq_len_sec
        self.min_seq_len_sec = min_seq_len_sec
        self.include_partial_samples = include_partial_samples
        self.zarr_files = zarr_files
        self.channels = channels
        self.channels_has_dim = any(isinstance(i, list) for i in self.channels)
        self.sample_seq_len_sec = sample_seq_len_sec
        self.sample_seq_len = sample_seq_len_sec * frequency
        self.frequency = frequency
        self.sample_stride_sec = sample_stride_sec
        self.clip_interpolations = clip_interpolations
        self.start_offset_sec = start_offset_sec
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.voltage_channels = voltage_channels if voltage_channels is not None else []
        self.y_time_column = y_time_column
        self.y_demographic_columns = y_demographic_columns if y_demographic_columns is not None else []
        self.y_demographic_norm_stats = y_demographic_norm_stats if y_demographic_norm_stats is not None else {}
        self.return_hypnogram_every_sec = return_hypnogram_every_sec
        self.hypnogram_padding_mask = hypnogram_padding_mask
        self.hypnogram_frequency = hypnogram_frequency
        self.trim_wake_epochs = trim_wake_epochs
        self.normalize_signals = normalize_signals
        
        if sample_df is None:
            print(f"Calculating samples with {sample_seq_len_sec} sec length and {sample_stride_sec} sec stride with {max_seq_len_sec} sec max")
            self.sample_df, self.total_samples = calculate_samples_mp(zarr_files, channels=channels, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, frequency=frequency, stride_sec=sample_stride_sec, start_offset_sec=start_offset_sec, include_partial_samples=include_partial_samples, min_seq_len_sec=min_seq_len_sec, constant_nan_tolerance=constant_nan_tolerance, constant_channels=constant_channels)
        else:
            assert 'file' in sample_df, "The `sample_df` must have a column `file` with the zarr file path."
            missing_in_zarrs = len(set(sample_df['file']) - set(zarr_files))
            missing_in_df = len(set(zarr_files) - set(sample_df['file']))
            if missing_in_zarrs > 0 or missing_in_df > 0:
                warnings.warn(f"There are {missing_in_zarrs} `zarr_files` not in the `sample_df` and {missing_in_df} files in the `sample_df` that arent in the `zarr_files`, they will be ignored.")
            sample_df = sample_df.loc[sample_df['file'].isin(zarr_files)].copy().reset_index(drop=True)
            self.sample_df = sample_df
            self.total_samples = len(sample_df)
        
        self.outcome_df = y_outcome_df
        self.y_outcome = y_outcome
        self.y_mapping_column = y_mapping_column
            
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample['file'], mode='r')
        sample_id = sample['file']
        signals = []
        channels = []
        if self.channels_has_dim:
            avail_channels = list(root_grp.array_keys())
            for p in self.channels:
                channels.append(next(x for x in p if x in avail_channels))
        else:
            channels = self.channels
        for channel in channels:
            channel_frequency = int(root_grp[channel].attrs.asdict()['signal_header']['sample_frequency'])
            channel_dimension = root_grp[channel].attrs.asdict()['signal_header']['dimension']
            if channel_frequency != self.frequency:
                start_idx, end_idx = int(sample['start_idx'] / self.frequency * channel_frequency), int(sample['end_idx'] / self.frequency * channel_frequency)
            else:
                start_idx, end_idx = sample['start_idx'], sample['end_idx']
            temp = np.array(root_grp[channel][start_idx:end_idx], dtype=np.float32)
            if channel_frequency != self.frequency:
                temp = resample_waveform(temp, channel_frequency, self.frequency, is_spo2=channel in SPO2_CHANNELS)
            if self.median_filter_kernel_size is not None:
                temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
            if self.butterworth_filters is not None and channel in self.butterworth_filters:
                freq_range = self.butterworth_filters[channel]
                btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                temp = iir_filter(temp, freq_range=freq_range, btype=btype, fs=self.frequency)
            if channel in self.voltage_channels:
                if channel_dimension.lower() == 'mv':
                    temp = temp * 1e3
            if self.clip_interpolations is not None and channel in self.clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
            else:
                temp = interpolate_nan_clip(temp)
            if self.normalize_signals:
                temp = iqr_normalization(temp, is_spo2=channel in SPO2_CHANNELS)
            signals.append(temp)
        X = torch.from_numpy(np.stack(signals, dtype=np.float32))
        if self.trim_wake_epochs and 'hypnogram' in root_grp.array_keys():
            if self.hypnogram_frequency == self.frequency:
                hypnogram = torch.from_numpy(np.array(root_grp['hypnogram'][sample['start_idx']:sample['end_idx']], dtype=np.int64))
            else:
                hyp_start_idx = int((sample['start_idx'] / self.frequency) * self.hypnogram_frequency)
                hyp_end_idx = int((sample['end_idx'] / self.frequency) * self.hypnogram_frequency)
                hypnogram = np.array(root_grp['hypnogram'][hyp_start_idx:hyp_end_idx], dtype=np.int64)
                hypnogram = torch.from_numpy(hypnogram)
        
            expected_hyp_len = int((X.shape[-1] / self.frequency) * self.hypnogram_frequency)
            if hypnogram.shape[-1] < expected_hyp_len:
                hypnogram = F.pad(hypnogram, (0, expected_hyp_len - hypnogram.shape[-1]), 'constant', value=self.hypnogram_padding_mask)
            
            samples_per_epoch = int(self.return_hypnogram_every_sec * self.hypnogram_frequency)
            hypnogram = hypnogram[::samples_per_epoch]
            hypnogram[hypnogram>5] = self.hypnogram_padding_mask
        Y = torch.tensor(self.outcome_df.loc[(self.outcome_df[self.y_mapping_column] == sample_id), self.y_outcome].T.values, dtype=torch.int64).squeeze(-1)
        
        if self.trim_wake_epochs and 'hypnogram' in root_grp.array_keys():
            resampled_hypnogram_length = int(self.return_hypnogram_every_sec * self.frequency)
            hypnogram = trim_wake_epochs_from_hypnogram(hypnogram, padding_mask=self.hypnogram_padding_mask)
            X = remove_wake_epochs_from_signals(X, hypnogram=hypnogram, resampled_hypnogram_length=resampled_hypnogram_length, padding_mask=self.hypnogram_padding_mask)
        if self.y_time_column is not None:
            time = self.outcome_df.loc[(self.outcome_df[self.y_mapping_column] == sample_id), self.y_time_column].values.T
            time = torch.tensor(time, dtype=torch.float32).squeeze(-1)
            if self.y_demographic_columns is not None and len(self.y_demographic_columns) > 0:
                demographics = self.outcome_df.loc[(self.outcome_df[self.y_mapping_column] == sample_id), self.y_demographic_columns].values
                for i, col in enumerate(self.y_demographic_columns):
                    if col in self.y_demographic_norm_stats:
                        mean_std = self.y_demographic_norm_stats[col]
                        demographics[:, i] = (demographics[:, i] - mean_std['mean']) / mean_std['std']
                demographics = torch.tensor(demographics, dtype=torch.float32).squeeze(0)
                return X, Y, time, demographics
            else:
                return X, Y, time
        else:
            return X, Y


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()